# Notebook 02: Traditional ML Baseline (handcrafted features + SVM)

**Model A** classifies 256 px tiles as *defect* (>= 5% corrosion) or *sound* (<= 0.5%). Features: Lab colour relative to the whole image, LBP texture and gradient statistics (47 values). Hyperparameters are chosen by **scene-grouped cross-validation** on the training split, using the one-standard-error rule to prefer the most regularised model.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from src.utils.config import load_config
from src.data.split_dataset import load_split_manifest
from src.models.baseline import (HullDefectBaselineClassifier, prepare_patch_dataset,
                                 select_hyperparameters_by_group_cv)
from src.models.evaluate import evaluate_baseline_on_patches

config = load_config()
splits = load_split_manifest('../data/processed/split_manifest_mavecodd.json')
patch_cfg = {'long_side': 1024, 'patch_size': 256, 'defect_min_frac': 0.05, 'sound_max_frac': 0.005}
print({k: len(v) for k, v in splits.items()})


## 1. Patches, features and grouped cross-validation


In [ ]:
train_imgs, train_y, meta = prepare_patch_dataset(splits['train'], return_meta=True, **patch_cfg)
print(f'Training patches: {len(train_imgs)} (defect {train_y.sum()}, sound {(train_y == 0).sum()})')

X_train = HullDefectBaselineClassifier().extract_batch_features(train_imgs, [m['context'] for m in meta])
cv = select_hyperparameters_by_group_cv(X_train, train_y, [m['scene_id'] for m in meta],
                                        c_values=[0.03, 0.1, 0.3, 1, 3, 10], pca_values=[None, 8, 16])
print('Chosen:', cv['chosen'])

model = HullDefectBaselineClassifier(use_pca=cv['chosen']['pca_components'] is not None,
                                     pca_components=cv['chosen']['pca_components'] or 16,
                                     svm_c=cv['chosen']['C']).fit(X_train, train_y)


## 2. Train vs validation vs test (overfitting check)


In [ ]:
for split in ['train', 'val', 'test']:
    r = evaluate_baseline_on_patches(model, splits[split], patch_cfg)
    print(f"{split:5s} balanced acc {r['balanced_accuracy']:.3f}  recall {r['recall']:.3f}  "
          f"precision {r['precision']:.3f}  {r['confusion_matrix']}")
